In [ ]:
# Importante para poder utilizar los datasets de Hugging Face

!pip install datasets

In [ ]:
# Los import necesarios 
import pandas as pd 

# Librerias para realizar graficos
import seaborn as sns
import matplotlib.pyplot as plt

# Para poder cargar los datasets
from datasets import load_dataset 

# Herramientas para el procesamiento y modelamiento de los datos
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Cargar el dataset
# Esta etapa sirve para analizar las caracteristicas de los datos, 
# asi como evaluar la calidad de los mismos

# El dataset seleccionado es: AiresPucrs/german-credit-data
dataset = load_dataset("AiresPucrs/german-credit-data", split="train")

df = dataset.to_pandas()

df.head()

In [ ]:
# La columna objetivo es: Risk, que nos indica si el riesgo es bueno o malo.
# Good: un buen credito, que tiene alta probabilidad de ser pagado
# Bad: un mal credito, que tiene alta probabilidad de no ser pagado.

# En este caso, se esta utilizando un set de datos, 
# pero se podria usar el historico de creditos de una entidad bancaria.
# Si fuera el caso, lo mas probable es que se tenga que ANONIMIZAR la data.

df['Risk'] = df['Risk'].str.strip().str.lower()  # eliminar espacios y pasar a minúsculas
df['Risk'] = df['Risk'].map({'good': 1, 'bad': 0})
df['Risk'] = df['Risk'].astype(int)

df.head()

In [ ]:
# Exploración
df.info()
df.head()
df.describe()

sns.countplot(x='Risk', data=df); 
plt.title('Distribución riesgo crediticio'); 
plt.show()

In [ ]:
# Preprocesamiento
num_cols = ['Age','Job','Credit amount','Duration']
cat_cols = ['Sex','Housing','Saving accounts','Checking account','Purpose']

# Reglas de Ingenieria de Caracteristicas
preprocessor = ColumnTransformer([
  ('num', StandardScaler(), num_cols),
  ('cat', OneHotEncoder(drop='first'), cat_cols)
])

In [ ]:
# Modelo del pipeline
pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42)),
])

# Separamos la variable objetivo de la variable resultado
X = df.drop("Risk", axis=1)
y = df["Risk"]

# Separamos los datos, en datos de entrenamiento y evaluacion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
# Entrenamiento basico
pipeline.fit(X_train, y_train)

# Estos son los resultados que obtendria, con mi modelo entrenado
y_pred = pipeline.predict(X_test)
print("Precision del modelo: ", accuracy_score(y_test, y_pred)) # Compara los resultados pronosticados vs los resultados reales
print(classification_report(y_test, y_pred))

In [ ]:
# Visualizacion del arbol
clf = pipeline.named_steps["clf"]
feature_names = num_cols + list(pipeline.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(cat_cols))

plt.figure(figsize = (20, 10))
plot_tree(clf, feature_names= feature_names, class_names=["Bad", "Good"], filled = True)

plt.show()

In [ ]:
# En este caso, solo se limita la visualizacion
# Si se quiere limitar el entrenamiento: 
# DecisionTreeClassifier(max_depth=5, random_state=42)
# Esto reduce el overfitting y hace el árbol más interpretable.

plt.figure(figsize = (20, 10))
plot_tree(clf, 
          feature_names= feature_names, 
          class_names=["Bad", "Good"], 
          filled = True, 
          max_depth = 2, # Limita la visualizacion a 2 niveles
         )

plt.show()


In [ ]:
# Optimización con GridSearch
# En este caso, se realizan todas las combinaciones posibles de las variables de param_grid
# Con cada combinacion, se entrena y evalua el modelo
# Nos indica la mejor combinacion de parametros

param_grid = {
  'clf__max_depth': [3,5,7],
  'clf__min_samples_split': [2,5,10],
  'clf__criterion': ['gini','entropy']
}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
print("Mejores params:", grid.best_params_, "→ CV score:", grid.best_score_)

In [ ]:
# Evaluacion Final
# Utilizando los parametros mas adecuados

best = grid.best_estimator_
yb = best.predict(X_test)
print("Accuracy optimizado:", accuracy_score(y_test, yb))
print(classification_report(y_test, yb))

In [ ]:
# Visualización árbol optimizado
opt_clf = best.named_steps['clf']
plt.figure(figsize=(20,10))
plot_tree(opt_clf, feature_names=feature_names, class_names=['Bad','Good'], filled=True)
plt.show()